## 默认离线构造路线

本Notebook新跑三对构造数据主流程并保存一幅新图；不因真实manifest存在而选择真实分支。G01权利/主体/用途unknown，G05/A10-B未授权；A10-A设计及A10-A.1技术协议接受不解除执行闸门。

[本语言逐本运行模式与准备条件](../../../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

# 项目5：前推配对与执行账本

前向总回报序列与未复权执行价格分开；504观察形成、126观察交易，参数冻结。入场/退出/止损阈值为±2/±.5/±4。v2选择为EG p<.05且beta>0，限定多空相对价值；冻结前已看构造结果，不能说是构造实验的纯事前注册。

账本区分信号/成交、配对/独立部分成交、开盘mark、ACT/360融资、借券与股息。5%已知ADV20只是未校准的事前预算，不保证实际参与率。对账通过不等于对所有输入证明零泄漏。

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("p05_pairs_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import json, hashlib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
root = ROOT / "projects/project05"
for path in (root, root / "src"): sys.path.insert(0,str(path))
import p05_constructed
import p05_pairs as pp
import run_project05


## 1. 配置与构造输入

读preregistration_v2.json及字节hash，代码明确调用load_inputs('constructed', prereg)。use_real=False不是许可开关，真实manifest冻结也不自动授权。

In [ ]:
prereg_path = root / 'configs' / 'preregistration_v2.json'
prereg_bytes = prereg_path.read_bytes()
prereg = json.loads(prereg_bytes)
prereg_digest = hashlib.sha256(prereg_bytes).hexdigest()
manifest_path = root / 'data_manifest.json'

# Always choose the offline constructed branch; real-data execution needs separate permission.
use_real = False
panel_df, rates_df, pairs, meta = run_project05.load_inputs('constructed', prereg)
print(f'CONSTRUCTED teaching panel ({meta["data_class"]}): {len(pairs)} pairs; real route SKIPPED')


## 2. 新算主流程

三对构造数据执行前推形成/交易，比较screened与ungated。选择为p<.05且beta>0；BH q单列，不替换选择规则。不是完整成本/阈值网格。

In [ ]:
results = run_project05.run_all(panel_df, rates_df, pairs, prereg)
print(f'Walk-forward completed: {len(results["windows"])} rolling windows.')
print(f'Total formation tests: {len(results["formations"])}')
print(f'EG p < 0.05: {int(results["formations"]["eg_pass"].sum())}; selected (p < 0.05 and beta > 0): {int(results["formations"]["selected"].sum())}')

## 3. 形成期表

核样本数、alpha、beta、spread标准差、半衰期、水平相关、EG统计/p、窗内BH q及EG-pass/selected两个旗标；统计筛选不等于盈利。

In [ ]:
df_form = results['formations'][[
    'pair', 'window', 'n', 'alpha', 'beta', 'spread_sd', 'half_life_days',
    'corr_level', 'eg_stat', 'eg_pvalue', 'qvalue_bh_window', 'eg_pass', 'selected'
]].copy()
print(df_form.to_string(index=False))

## 4. 图中信号与成交的两个时钟

CA_CB代表窗2的曲线横轴为signal_date、纵轴为该日z。成交标记横轴放fill_date，纵轴却取对应前一signal_date的z，表示延迟成交与原信号连接，不是成交日实测z。共用图代码保持；新图写RUN_DIR/pairs_spread_CONSTRUCTED.png。

In [ ]:
target_pair = 'CA_CB'
w_target = results['windows'][2]  # Representative window 2
form_target = results['formations'][(results['formations']['pair'] == target_pair) & (results['formations']['window'] == w_target.k)].iloc[0]

P = results['P']
log_tr = np.log(pp.total_return_index(P))
sig_dates = w_target.signal_dates
s = log_tr.loc[sig_dates, form_target['y']] - form_target['alpha'] - form_target['beta'] * log_tr.loc[sig_dates, form_target['x']]
z = (s - form_target['spread_mean']) / form_target['spread_sd']

fig, ax = plt.subplots(figsize=(10, 4.5), dpi=150)
ax.plot(z.index, z.values, color='steelblue', linewidth=1.8, label='Spread z-score')
ax.axhline(2.0, color='crimson', linestyle='--', label='Entry (+2.0)')
ax.axhline(0.5, color='darkorange', linestyle=':', label='Exit (+0.5)')
ax.axhline(0.0, color='gray', linestyle='-', linewidth=0.8)
ax.axhline(-0.5, color='darkorange', linestyle=':', label='Exit (-0.5)')
ax.axhline(-2.0, color='crimson', linestyle='--', label='Entry (-2.0)')

orders = results['orders']
sub_o = orders[(orders['pair'] == target_pair) & (orders['gate'] == 'screened')]
if len(sub_o):
    for _, row_o in sub_o.iterrows():
        sd = pd.Timestamp(row_o['signal_date'])
        if sd in z.index:
            zv = z.loc[sd]
            color = 'green' if row_o['kind'] == 'entry' else 'purple'
            marker = '^' if (row_o['kind'] == 'entry' and zv < 0) else ('v' if row_o['kind'] == 'entry' else 'o')
            ax.scatter(pd.Timestamp(row_o['fill_date']), zv, color=color, marker=marker, s=70, zorder=5)

ax.set_title(f'Pair {target_pair} Spread Dynamics & Ledger Execution (Window {w_target.k})\n[CONSTRUCTED — not market evidence]', fontsize=11, fontweight='bold')
ax.set_xlabel('Trading Date (YYYY-MM)')
ax.set_ylabel('Spread z-score (SD)')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
fig.savefig(RUN_DIR / "pairs_spread_CONSTRUCTED.png", dpi=130)
plt.show()

## 5. 现金对账

现金−(初财富+利息−借券+股息−交易流−佣金)应在容差内。点差/冲击已进fill_price，不可再扣一次。

In [ ]:
reconciliation = []
for (pair, gate), led in results['ledgers'].items():
    reconciliation.append({
        'pair': pair,
        'gate': gate,
        'interest': led['totals']['interest'],
        'borrow': led['totals']['borrow'],
        'dividends': led['totals']['dividends'],
        'commission': led['totals']['commission'],
        'spread_cost': led['totals']['spread_cost'],
        'impact_cost': led['totals']['impact_cost'],
        'cash_residual': led['cash_residual']
    })
df_rec = pd.DataFrame(reconciliation)
print('LEDGER RECONCILIATION SUMMARY:')
print(df_rec[['pair', 'gate', 'interest', 'borrow', 'dividends', 'commission', 'cash_residual']].to_string(index=False))
assert (df_rec['cash_residual'].abs() < 1e-7).all(), 'Non-zero cash residual detected!'

## 6. 构造绩效

读总/年化收益、波动、相对每日现金计息的超额Sharpe、回撤、开闭仓数、持有日数、换手口径与现金残差。不是市场证据或真实路线授权。

In [ ]:
metrics_df = run_project05.metrics_table(results)
print('HEADLINE METRICS TABLE [CONSTRUCTED — NOT MARKET EVIDENCE]:')
cols = ['pair', 'gate', 'n_returns', 'total_return', 'ann_return', 'ann_vol', 'sharpe_excess', 'max_drawdown', 'closed_trades', 'open_trades', 'win_rate', 'mean_holding_days', 'turnover_one_sided_ann', 'cash_residual']
print(metrics_df[cols].to_string(index=False))